# Triton IR：同一个加法 kernel 的不同表示

**IR = Intermediate Representation，中间表示。**
配套讲义：`docs/TRITON_IR.md`。
这份实验复用课程 Vector Add，不重写 kernel，不下载模型，不衡量性能。
它是Day 17的补充预览，不因执行参考实验自动改变个人进度；当前学习位置见 [进度记录](../docs/LEARNING_PROGRESS.md)。

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime, timezone

root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/triton_ir.py').is_file():
    raise RuntimeError('Open from the repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import torch
from learning.triton_ir import inspect_vector_add
print('CUDA available:', torch.cuda.is_available())

## 1. 编译并执行一次真实 Vector Add

N=1003，BLOCK=256，因此需要4个program，最后一个必须Mask尾部。
实验先验证输出，再读取实际的 `CompiledKernel.asm`；不从字符串伪造编译结果。
没有CUDA时明确跳过，不算通过。每次保存到新的目录，避免覆盖上次输出。

In [ ]:
report = None
if torch.cuda.is_available():
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    destination = Path('logs') / f'triton-ir-{stamp}'
    report = inspect_vector_add(output=destination)
    print('Triton:', report['triton_version'], 'GPU:', report['gpu'])
    print('编译产物:', report['available_asm_keys'])
    print('数值验证:', report['numerical_check'])
    print('寄存器/线程:', report['registers_per_thread'])
    print('Shared memory bytes:', report['shared_memory_bytes'])
    print('保存目录:', destination)
else:
    print('需要NVIDIA CUDA，IR实验未执行；不是通过。')

## 2. 先看 TTIR，再看 TTGIR，最后看 PTX

TTIR关注张量操作，TTGIR增加GPU布局信息，PTX更接近GPU虚拟指令。
这里仅打印每份文件的前16行；完整内容保存在日志目录。
注意：PTX不是最终机器指令，cubin也不是需要逐行阅读的文本IR。

In [ ]:
if report is not None:
    for stage in ('ttir', 'ttgir', 'llir', 'ptx'):
        print('\n===', stage, '===')
        print(report['preview'].get(stage, 'This stage is unavailable in this environment.'))

## 自测与答案要点

- IR全称？Intermediate Representation，中间表示，不是模型中间激活。
- 为什么分多层？逐步从张量语义降低到布局、访存和目标指令，便于优化与代码生成。
- TTGIR多了什么？相较TTIR有GPU分布/布局等信息，具体看目标版本产物。
- PTX是机器码吗？不是，它是NVIDIA虚拟ISA；本路径再由ptxas生成cubin。
- 产物存在能证明更快吗？不能，只证明编译/执行，速度还需独立Benchmark/Profiler。

修改BLOCK前先预测grid和尾部Mask，再比较IR；不要把文本更短直接解释成性能更好。